In [ ]:
pip install -U bitsandbytes

In [ ]:
pip install librosa transformers torchaudio

In [ ]:
!pip install msclap

In [ ]:
import zipfile
import os
import shutil

kaggle_input_dir = '/kaggle/input/splits-without-augmentation-updated226'
kaggle_output_dir = '/kaggle/working/data'

shutil.copytree(kaggle_input_dir, kaggle_output_dir, dirs_exist_ok=True)


In [ ]:
# Generate a csv file to record the file name and the emotion label
def generate_csv(input_dir, output_dir, csv_name):
    # Get the file names of the audio files
    file_names = os.listdir(input_dir)
    # Get the emotion labels of the audio files
    emotion_labels = [file_name.split('_')[-1].split('.')[0] for file_name in file_names]
    # Create a dictionary to store the file names and the emotion labels
    data = {'file_name': file_names, 'audio_emotion_label': emotion_labels}
    # Create a DataFrame to store the data
    df = pd.DataFrame(data)
    # Save the DataFrame to a csv file
    df.to_csv(os.path.join(output_dir, csv_name), index=False)

In [ ]:
import pandas as pd
splits_list = ['train', 'val', 'test']
formats_list = ['audio']
for split in splits_list:
    for format in formats_list:
        # Generate the input directory
        input_dir = f'/kaggle/working/data/splits/MELD/{split}/{format}/'
        # Generate the output directory
        output_dir = f'/kaggle/working/data/splits/MELD/{split}/'
        # Generate the csv file
        csv_name = f'{split}.csv'
        generate_csv(input_dir, output_dir, csv_name)

In [ ]:
# 在Kaggle Notebook中运行
!wget https://huggingface.co/microsoft/msclap/resolve/main/CLAP_weights_2023.pth

In [ ]:
from msclap import CLAP
from pathlib import Path
import torch

# 手动指定存储目录
model_dir = Path("/kaggle/working/output")
model_dir.mkdir(parents=True, exist_ok=True)

url = 'https://huggingface.co/microsoft/msclap/resolve/main/CLAP_weights_2023.pth'
model_file = model_dir / url.split('/')[-1]

# 这里可以添加下载模型文件的代码，如果文件不存在
if not model_file.exists():
    import requests
    response = requests.get(url)
    with open(model_file, 'wb') as f:
        f.write(response.content)

model = CLAP(model_file, version = '2023', use_cuda=torch.device('cuda'))
print(model.get_audio_embeddings(["/kaggle/input/splits-without-augmentation-updated226/splits/MELD/train/audio/dia0_utt0_neutral.wav"]))

In [ ]:
model_file

In [ ]:
# your huggingface token here

In [ ]:
# ** 处理文本并生成文本情绪标签**
def predict_text_emotion(tokenizer, emotion_classifier, transcript):
    """
    改进版情感预测函数

    参数:
        tokenizer: 分词器
        emotion_classifier: 加载的情感分类器
        transcript: 需要分析的文本

    返回:
        str: 情绪标签（小写）
    """
    # 确保输入为字符串
    if not isinstance(transcript, str) or len(transcript.strip()) == 0:
        return "neutral"

    # 分词处理
    inputs = tokenizer(transcript, return_tensors="pt").to(emotion_classifier.device)

    # 模型推理
    with torch.no_grad():
        outputs = emotion_classifier(**inputs)

    # 获取预测结果
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()

    # 获取标签映射
    label_mapping = emotion_classifier.config.id2label
    predicted_emotion = label_mapping[predicted_class_id]
    print(transcript)
    print(predicted_emotion)
    return predicted_emotion.lower()

     

In [ ]:
# for文本标签测试
## 加载模型和分词器
# model_name = "j-hartmann/emotion-english-distilroberta-base"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# emotion_classifier = AutoModelForSequenceClassification.from_pretrained(model_name)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# emotion_classifier.to(device)

# # 准备测试文本
# test_texts = [
#     "This is an amazing !",
#     "graphic design all my creative filmmaking all my creative outlets that",
#     "How could you do this? I'm so angry!",
#     "Wow!",
#     "There's a strange noise in the dark.",
#     "well i have this theory that you could make me feel bad about myself only if i let you",
#     "The weather is nice today."
# ]

# # 进行情感预测并打印结果
# for text in test_texts:
#     emotion = predict_text_emotion(tokenizer, emotion_classifier, text)
#     print(f"Text: {text}")
#     print(f"Predicted emotion: {emotion}")
#     print("-" * 50)

In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM


# # model_name = "meta-llama/Llama-2-7b-chat-hf"
# # tokenizer = AutoTokenizer.from_pretrained(model_name)

# from transformers import BitsAndBytesConfig

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
# llm_model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B").to(device)

# # 示例文本
# transcript = "This is a really amazing day!."

# # 调用函数进行预测
# result = predict_text_emotion(tokenizer, llm_model, transcript)
# print("预测的情绪类别:", result)

In [ ]:
# # 调用函数进行预测
# transcript = "This is a really amazing day! "

# result = predict_text_emotion(tokenizer, llm_model, transcript)
# print("预测的情绪类别:", result)

In [ ]:
# # for hubert model test
# import torch
# import torchaudio
# from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

# # 加载特征提取器和模型
# feature_extractor = AutoFeatureExtractor.from_pretrained("superb/hubert-large-superb-er")
# model = AutoModelForAudioClassification.from_pretrained("superb/hubert-large-superb-er")

# # 检查是否有可用的 GPU，如果有则将模型移动到 GPU 上
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# def load_and_preprocess_audio(audio_file_path):
#     """
#     加载音频文件并进行预处理
#     :param audio_file_path: 音频文件的路径
#     :return: 预处理后的音频数据
#     """
#     # 加载音频文件
#     speech, sampling_rate = torchaudio.load(audio_file_path)

#     # 若音频采样率与模型要求的采样率不一致，进行重采样
#     if sampling_rate != feature_extractor.sampling_rate:
#         resampler = torchaudio.transforms.Resample(sampling_rate, feature_extractor.sampling_rate)
#         speech = resampler(speech)

#     # 取单声道音频
#     speech = speech.squeeze().numpy()

#     # 对音频进行预处理
#     inputs = feature_extractor(speech, return_tensors="pt", sampling_rate=feature_extractor.sampling_rate)
#     inputs = {k: v.to(device) for k, v in inputs.items()}
#     return inputs

# def predict_emotion(inputs):
#     """
#     使用模型进行情感预测
#     :param inputs: 预处理后的音频输入
#     :return: 预测的情感标签
#     """
#     # 进行推理
#     with torch.no_grad():
#         outputs = model(**inputs)

#     # 获取预测的 logits
#     logits = outputs.logits

#     # 获取预测的类别索引
#     predicted_class_idx = logits.argmax(-1).item()

#     # 获取类别标签
#     predicted_emotion = model.config.id2label[predicted_class_idx]
#     if 
#     print(type(predicted_emotion), predicted_emotion)
#     return predicted_emotion

# if __name__ == "__main__":
#     # 替换为你的音频文件路径
#     audio_file = "/kaggle/input/splits-without-augmentation-updated226/splits/DAIC_WOZ/train/audio/302_20.wav"

#     # 加载并预处理音频
#     inputs = load_and_preprocess_audio(audio_file)

#     # 进行情感预测
#     emotion = predict_emotion(inputs)

#     print(f"Predicted emotion: {emotion}")

In [ ]:
# model.config.id2label

In [ ]:
# 映射字典，将简短标签映射到完整标签
SHORT_TO_FULL_EMOTION_MAP = {
    'neu': 'neutral',
    'hap': 'joy',
    'ang': 'anger',
    'sad': 'sadness'
}

def predict_audio_emotion_hubert(audio_path, hubert_processor, hubert_model):
    """
    使用superb/hubert-large-superb-er模型进行音频情绪预测
    参数:
        audio_path: 待预测音频路径
        hubert_processor: Hubert模型的处理器
        hubert_model: Hubert模型
    返回:
        predicted_emotion: 预测情绪
    """
    try:
        audio, sr = load_and_resample_audio(audio_path, hubert_processor.feature_extractor.sampling_rate)
        if audio is None:
            return None
        inputs = hubert_processor(audio.squeeze(0).numpy(), return_tensors="pt", sampling_rate=hubert_processor.feature_extractor.sampling_rate).to(DEVICE)
        with torch.no_grad():
            outputs = hubert_model(**inputs)
        logits = outputs.logits
        predicted_class_idx = logits.argmax(-1).item()
        short_emotion = hubert_model.config.id2label[predicted_class_idx].lower()
        # 进行标签映射
        full_emotion = SHORT_TO_FULL_EMOTION_MAP.get(short_emotion)
        if full_emotion is None:
            print(f"未找到匹配的完整情绪标签，简短标签为: {short_emotion}")
            return short_emotion
        return full_emotion
    except Exception as e:
        print(f"Hubert model prediction failed: {str(e)}")
        return None

In [ ]:
# -*- coding: utf-8 -*-
"""
音频情绪伪标签和文本情绪伪标签生成
主要功能：
1. 使用Wav2Vec 2.0和CLAP计算FAD分数生成音频伪标签1
2. 使用hubert生成音频伪标签2
3. 生成文本的伪标签
4. 保存带置信度的最终标签（其中2者相同即可）
"""

# ==================== 环境设置 ====================
import os
import re
import torch
import librosa
import numpy as np
import pandas as pd
from scipy.linalg import sqrtm
import shutil
from pathlib import Path
from tqdm import tqdm

# 深度学习相关库
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2Model,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoFeatureExtractor,
    AutoModelForAudioClassification
)

# CLAP相关库
try:
    from msclap import CLAP
except ImportError:
    raise ImportError("请安装msclap: pip install msclap")

# ==================== 常量定义 ====================
EMOTIONS = ["neutral", "joy", "sadness", "anger", "surprise", "fear", "disgust"]
FEATURE_WEIGHTS = {'wav2vec2': 0.6, 'clap': 0.4}  # 特征权重可调整
CLAP_SAMPLE_RATE = 44100  # CLAP要求的采样率
WAV2VEC_SAMPLE_RATE = 16000  # Wav2Vec2要求的采样率
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 映射字典，将简短标签映射到完整标签
SHORT_TO_FULL_EMOTION_MAP = {
    'neu': 'neutral',
    'hap': 'joy',
    'ang': 'anger',
    'sad': 'sadness'
}

# ==================== 模型加载 ====================
def load_models():
    """
    加载所有需要的模型
    返回:
        wav2vec2_processor: Wav2Vec2的处理器
        wav2vec2_model: Wav2Vec2模型
        clap_model: CLAP模型
        tokenizer: 文本tokenizer
        emotion_classifier: 情感分类模型
        hubert_feature_extractor: Hubert模型的特征提取器
        hubert_model: Hubert模型
    """
    # Wav2Vec2模型加载
    print("Loading Wav2Vec2 model...")
    wav2vec2_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
    wav2vec2_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)

    # CLAP模型加载
    print("Loading CLAP model...")
    clap_model = CLAP(
        "CLAP_weights_2023.pth",  # 需要提前下载到工作目录
        version='2023',
        use_cuda=torch.device('cuda')
    )

    # 分词器加载
    tokenizer = AutoTokenizer.from_pretrained("j-hartmann/emotion-english-distilroberta-base")
    # 情感分类模型加载
    emotion_classifier = AutoModelForSequenceClassification.from_pretrained("j-hartmann/emotion-english-distilroberta-base")
    emotion_classifier.to(DEVICE)

    # 加载superb/hubert-large-superb-er模型
    print("Loading Hubert model...")
    hubert_feature_extractor = AutoFeatureExtractor.from_pretrained("superb/hubert-large-superb-er")
    hubert_model = AutoModelForAudioClassification.from_pretrained("superb/hubert-large-superb-er").to(DEVICE)

    return wav2vec2_processor, wav2vec2_model, clap_model, tokenizer, emotion_classifier, hubert_feature_extractor, hubert_model

# ==================== 音频处理 ====================
def load_and_resample_audio(file_path, target_sr):
    """
    加载并重采样音频文件
    参数:
        file_path: 音频文件路径
        target_sr: 目标采样率
    返回:
        waveform: 重采样后的波形 (1, T)
        sr: 实际采样率
    """
    try:
        waveform, sr = librosa.load(file_path, sr=target_sr, mono=True)
        return torch.tensor(waveform).unsqueeze(0), sr
    except Exception as e:
        print(f"音频加载失败: {file_path} - {str(e)}")
        return None, None

def extract_embeddings(wav2vec2_processor, wav2vec2_model, clap_model, audio_path):
    """
    提取双模态音频特征
    参数:
        audio_path: 音频文件路径
    返回:
        wav2vec_emb: Wav2Vec2特征 (1, 768)
        clap_emb: CLAP特征 (1, 1024)
    """
    # Wav2Vec2特征提取
    try:
        # 加载并处理音频
        wav_input, _ = load_and_resample_audio(audio_path, WAV2VEC_SAMPLE_RATE)
        if wav_input is None:
            return None, None

        # 特征提取
        with torch.no_grad():
            inputs = wav2vec2_processor(
                wav_input.squeeze(0).numpy(),
                return_tensors="pt",
                sampling_rate=WAV2VEC_SAMPLE_RATE
            ).to(DEVICE)
            wav2vec_emb = wav2vec2_model(**inputs).last_hidden_state.mean(dim=1)
            wav2vec_emb = wav2vec_emb.cpu().numpy()
    except Exception as e:
        print(f"Wav2Vec2特征提取失败: {str(e)}")
        wav2vec_emb = None

    # CLAP特征提取
    try:
        # CLAP需要原始路径输入
        clap_emb = clap_model.get_audio_embeddings([audio_path], resample=True)
        clap_emb = clap_emb.cpu().numpy() if torch.is_tensor(clap_emb) else clap_emb
    except Exception as e:
        print(f"CLAP特征提取失败: {str(e)}")
        clap_emb = None

    return wav2vec_emb, clap_emb

# ==================== FAD计算 ====================
def compute_fad(emb_ref, emb_target, eps=1e-6):
    """
    计算Frechet Audio Distance (FAD)
    参数:
        emb_ref: 参考特征 (N, D)
        emb_target: 目标特征 (M, D)
        eps: 数值稳定系数
    返回:
        fad_score: FAD分数
    """
    # 确保输入是2维数组
    if emb_ref.ndim > 2:
        emb_ref = emb_ref.reshape(-1, emb_ref.shape[-1])
    if emb_target.ndim > 2:
        emb_target = emb_target.reshape(-1, emb_target.shape[-1])

    # 均值计算
    mu_ref = np.mean(emb_ref, axis=0)
    mu_target = np.mean(emb_target, axis=0)

    # 协方差计算
    sigma_ref = np.cov(emb_ref, rowvar=False) + eps * np.eye(emb_ref.shape[1])
    sigma_target = np.cov(emb_target, rowvar=False) + eps * np.eye(emb_target.shape[1])

    # 矩阵平方根计算
    sqrt_product = sqrtm(sigma_ref.dot(sigma_target))
    if np.iscomplexobj(sqrt_product):
        sqrt_product = sqrt_product.real

    # 计算FAD
    trace_term = np.trace(sigma_ref + sigma_target - 2 * sqrt_product)
    mean_term = np.sum((mu_ref - mu_target) ** 2)

    return mean_term + trace_term

# ==================== 情绪预测 ====================
def predict_audio_emotion(audio_path, labeled_embeddings, wav2vec2_processor, wav2vec2_model, clap_model):
    """
    基于FAD的音频情绪预测
    参数:
        audio_path: 待预测音频路径
        labeled_embeddings: 已标注的特征库
    返回:
        predicted_emotion: 预测情绪
        confidence: 置信度
    """
    # 特征提取
    wav2vec_emb, clap_emb = extract_embeddings(wav2vec2_processor, wav2vec2_model, clap_model, audio_path)
    if wav2vec_emb is None or clap_emb is None:
        return None, 0.0

    # 计算每个情绪的FAD分数
    emotion_scores = []
    for emotion in EMOTIONS:
        scores = []
        # Wav2Vec2分数
        if len(labeled_embeddings[emotion]['wav2vec2']) > 0:
            wav2vec_score = compute_fad(
                np.array(labeled_embeddings[emotion]['wav2vec2']),
                wav2vec_emb
            ) * FEATURE_WEIGHTS['wav2vec2']
            scores.append(wav2vec_score)

        # CLAP分数
        if len(labeled_embeddings[emotion]['clap']) > 0:
            clap_score = compute_fad(
                np.array(labeled_embeddings[emotion]['clap']),
                clap_emb
            ) * FEATURE_WEIGHTS['clap']
            scores.append(clap_score)

        emotion_scores.append(np.mean(scores))

    # 计算置信度
    min_score = np.min(emotion_scores)
    confidence = 1 / (1 + min_score)
    return EMOTIONS[np.argmin(emotion_scores)], confidence


def predict_audio_emotion_hubert(audio_path, hubert_feature_extractor, hubert_model):
    """
    使用superb/hubert-large-superb-er模型进行音频情绪预测
    参数:
        audio_path: 待预测音频路径
        hubert_feature_extractor: Hubert模型的特征提取器
        hubert_model: Hubert模型
    返回:
        predicted_emotion: 预测情绪
    """
    try:
        audio, sr = load_and_resample_audio(audio_path, hubert_feature_extractor.sampling_rate)
        if audio is None:
            return None
        inputs = hubert_feature_extractor(audio.squeeze(0).numpy(), return_tensors="pt", sampling_rate=hubert_feature_extractor.sampling_rate).to(DEVICE)
        with torch.no_grad():
            outputs = hubert_model(**inputs)
        logits = outputs.logits
        predicted_class_idx = logits.argmax(-1).item()
        short_emotion = hubert_model.config.id2label[predicted_class_idx].lower()
        # 进行标签映射
        full_emotion = SHORT_TO_FULL_EMOTION_MAP.get(short_emotion)
        if full_emotion is None:
            print(f"未找到匹配的完整情绪标签，简短标签为: {short_emotion}")
            return None
        return full_emotion
    except Exception as e:
        print(f"Hubert模型预测失败: {str(e)}")
        return None


def parse_emotion_from_filename(filename):
    """
    从文件名解析情绪标签（格式：XXX_emotion.wav）
    参数:
        filename: 文件名
    返回:
        情绪标签字符串
    """
    return filename.split('_')[-1].split('.')[0]

# ==================== 主处理流程 ====================
def process_datasets(meld_root, daic_root, output_root):
    """
    主处理函数
    参数:
        meld_root: MELD数据集路径
        daic_root: DAIC-WOZ数据集路径
        output_root: 输出路径
    """
    # 初始化模型
    wav2vec2_processor, wav2vec2_model, clap_model, tokenizer, emotion_classifier, hubert_feature_extractor, hubert_model = load_models()

    # 遍历每个数据集分割
    # for split in ['train', 'val', 'test']:
    for split in ['train']:

        print(f"\nProcessing {split} split...")

        # ===== 处理MELD标注数据 =====
        meld_audio_dir = os.path.join(meld_root, split, 'audio')
        meld_csv = os.path.join(meld_root, split, f"{split}.csv")

        # 构建特征库
        labeled_embeddings = {emo: {'wav2vec2': [], 'clap': []} for emo in EMOTIONS}
        meld_df = pd.read_csv(meld_csv)

        for _, row in tqdm(meld_df.iterrows(), desc="Processing MELD"):
            audio_path = os.path.join(meld_audio_dir, row['file_name'])
            wav2vec_emb, clap_emb = extract_embeddings(wav2vec2_processor, wav2vec2_model, clap_model, audio_path)

            if wav2vec_emb is not None:
                labeled_embeddings[row['audio_emotion_label']]['wav2vec2'].append(wav2vec_emb)
            if clap_emb is not None:
                labeled_embeddings[row['audio_emotion_label']]['clap'].append(clap_emb)

        # ===== 处理DAIC-WOZ数据 =====

        daic_audio_dir = os.path.join(daic_root, split, 'audio')
        daic_transcripts_dir = os.path.join(daic_root, split, 'text')
        output_csv = os.path.join(output_root, f"daic_{split}_labels.csv")
        results = []

        files = sorted(os.listdir(daic_audio_dir))[7600:] # 示例处理前50个
        for audio_file in tqdm(files, desc="Processing DAIC"):
            audio_path = os.path.join(daic_audio_dir, audio_file)
            transcript_path = os.path.join(daic_transcripts_dir, audio_file.replace('.wav', '.txt'))

            # 生成音频伪标签1
            audio_label_1, confidence = predict_audio_emotion(
                audio_path, labeled_embeddings,
                wav2vec2_processor, wav2vec2_model, clap_model
            )

            # 生成音频伪标签2
            audio_label_2 = predict_audio_emotion_hubert(audio_path, hubert_feature_extractor, hubert_model)

            # 生成文本伪标签
            with open(transcript_path, 'r', encoding='utf-8') as f:
                transcript = f.read()
            text_label = predict_text_emotion(tokenizer, emotion_classifier, transcript)

            # 确定最终标签
            labels = [audio_label_1, audio_label_2, text_label]
            unique_labels, counts = np.unique(labels, return_counts=True)
            if np.max(counts) >= 2:
                final_label = unique_labels[np.argmax(counts)]
            else:
                final_label = "uncertain"

            results.append({
                'file': audio_file,
                'transcript': transcript,
                'audio_predicted_1': audio_label_1,
                'audio_predicted_2': audio_label_2,
                'confidence': confidence,
                "text_predicted": text_label,
                "final_label": final_label
            })

        # 保存结果
        pd.DataFrame(results).to_csv(output_csv, index=False)
        print(f"Saved results to {output_csv}")

# ==================== 执行入口 ====================
if __name__ == "__main__":
    # 路径配置
    kaggle_input_dir = '/kaggle/input/splits-updated-223'
    kaggle_output_dir = '/kaggle/working/data'
    meld_root = "/kaggle/working/data/splits/MELD"
    daic_root = "/kaggle/working/data/splits/DAIC_WOZ"
    output_root = "/kaggle/working/output"

    # 准备数据
    shutil.copytree(kaggle_input_dir, kaggle_output_dir, dirs_exist_ok=True)
    shutil.rmtree(output_root, ignore_errors=True)
    os.makedirs(output_root, exist_ok=True)

    # 运行处理流程
    process_datasets(meld_root, daic_root, output_root)

In [ ]:
# /kaggle/working/data/splits/MELD/train/train.csv